# Stage III. Five-fold ensemble inference on the test images

Only the test images that passed the Stage I gate (19,154 of 58,689) reach Stage III. Each is
preprocessed exactly as in training (bounding-box crop, LAB-lightness CLAHE, longest side 512,
zero padding to 512x512, masks cropped with the same box), scored by the five fold models, and
the five-way probability vectors are averaged.

Because the CORN decomposition is invertible, the averaged vector is mapped back to
`(p1, p2, p3, p_g)` and decoded with the OOF-fitted thresholds; no re-inference is needed to
change the decode.

**Steps.**
1. Audit mask availability for the routed images.
2. Run batched, resumable ensemble inference (`lgdrg_test_predictions.csv`).
3. Verify the inversion on the saved vectors.
4. Decode with the OOF thresholds.

## 0. Configuration

In [ ]:
from pathlib import Path

# Test images routed to Stage III by the Stage I gate, and their Stage II masks.
#   outputs/stage1_gate/dr_from_test/<stem>.png|jpg          (stage1_screening/03)
#   outputs/stage2_masks/test/<lesion>/<stem><suffix>.png     (stage2_segmentation/04 and 08)
TEST_DR_DIR = Path("outputs/stage1_gate/dr_from_test")
TEST_MASK_ROOT = Path("outputs/stage2_masks/test")

OUT_ROOT = Path("outputs")
CKPT_DIR = OUT_ROOT / "stage3" / "checkpoints"
EVAL_DIR = OUT_ROOT / "stage3" / "eval"
PRED_CSV = EVAL_DIR / "lgdrg_test_predictions.csv"
THRESH_JSON = EVAL_DIR / "corn_thresholds.json"      # from notebook 02
EVAL_DIR.mkdir(parents=True, exist_ok=True)

import json
import numpy as np
import pandas as pd
import torch
from hierarchiretina.stage3.engine import LGDRGConfig

cfg = LGDRGConfig(out_dir=str(CKPT_DIR))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE, NUM_WORKERS = 8, 0

## 1. Mask audit

A missing mask is filled with zeros at load time without an error, so every lesion should be
found for 100% of the images before inference.

In [ ]:
from hierarchiretina.stage3.data import LESIONS, build_infer_frame

fdf = build_infer_frame(TEST_DR_DIR, {l: TEST_MASK_ROOT / l for l in LESIONS})
print(f"images: {len(fdf):,}")

## 2. Ensemble inference

One row per image: `p_grade_1..5` (mean of the five members), the argmax grade, and a status
flag. Results are cached every 50 batches; re-running resumes.

In [ ]:
from hierarchiretina.stage3.engine import load_fold_models, run_test_inference

if PRED_CSV.exists():
    pred = pd.read_csv(PRED_CSV)
else:
    models = load_fold_models(cfg, CKPT_DIR, device=DEVICE)
    pred = run_test_inference(models, fdf, cfg, PRED_CSV, BATCH_SIZE, NUM_WORKERS)
    del models
pred = pred[pred.status == "OK"].copy()
print(f"predictions: {len(pred):,}")

## 3. Inversion check

Recover `(p1, p2, p3, p_g)` from the saved vectors, rebuild the vectors, and confirm that the
argmax reproduces the saved grade.

In [ ]:
from hierarchiretina.stage3.corn import corn_from_full, full_from_corn, decode_consecutive

PC = [f"p_grade_{k}" for k in range(1, 6)]
P5 = pred[PC].values
pp = corn_from_full(P5)
err = np.abs(full_from_corn(*pp) - P5).max()
agree = (P5.argmax(1) + 1 == pred.pred_grade.values).mean()
print(f"max |full(invert(P)) - P| = {err:.2e};  argmax reproduces pred_grade: {agree:.2%}")

## 4. Decode with the OOF thresholds

In [ ]:
th = json.load(open(THRESH_JSON))
pred["grade_tuned"] = decode_consecutive(*pp, th["t"], th["tg"])
print(f"t = {th['t']}, t_g = {th['tg']}")
pd.DataFrame({"argmax": pred.pred_grade.value_counts().sort_index(),
              "tuned": pred.grade_tuned.value_counts().sort_index()})